In [10]:
import os
import subprocess
import sys

if os.path.exists('yolov5'):
    print("YOLOv5 directory already exists!")
else:
    print("Cloning YOLOv5 repository...")
    try:
        result = subprocess.run(['git', 'clone', 'https://github.com/ultralytics/yolov5'], 
                                capture_output=True, text=True, cwd='.')
        if result.returncode == 0:
            print("YOLOv5 cloned")
        else:
            print(f"Git clone failed: {result.stderr}")
    except Exception as e:
        print(f"Error: {e}")

if os.path.exists('yolov5'):
    os.chdir('yolov5')
    if os.path.exists('requirements.txt'):
        try:
            result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], 
                                    capture_output=True, text=True)
            if result.returncode == 0:
                print("Requirements installed")
            else:
                print(f"packages may have failed: {result.stderr}")
        except Exception as e:
            print(f"Error installing requirements: {e}")


Cloning YOLOv5 repository...
YOLOv5 cloned
Requirements installed


In [11]:
import urllib.request
import os
from pathlib import Path

def download_file(url, filename, directory):
    """Download file with error handling into specified directory"""
    try:
        os.makedirs(directory, exist_ok=True)
        filepath = os.path.join(directory, filename)
        print(f"Downloading {filename} to {directory}...")
        urllib.request.urlretrieve(url, filepath)
        if os.path.exists(filepath):
            size = os.path.getsize(filepath)
            if size > 0:
                return True
            else:
                return False
        else:
            return False
    except:
        return False

dataset_dir = "dataset"
print(f"Files will be downloaded to: {os.path.abspath(dataset_dir)}")

dataset_files = {
    'labels.zip': 'https://github.com/ch-hristov/p-id-symbols/raw/main/labels.zip',
    'images.zip': 'https://github.com/ch-hristov/p-id-symbols/raw/main/images.zip', 
    'train.txt': 'https://github.com/ch-hristov/p-id-symbols/raw/main/train.txt',
    'val.txt': 'https://github.com/ch-hristov/p-id-symbols/raw/main/val.txt',
    'dataset.yaml': 'https://github.com/ch-hristov/p-id-symbols/raw/main/dataset.yaml',
    'best.pt': 'https://github.com/ch-hristov/p-id-symbols/raw/main/best.pt'
}

successful_downloads = []
failed_downloads = []

for filename, url in dataset_files.items():
    if download_file(url, filename, dataset_dir):
        successful_downloads.append(filename)
    else:
        failed_downloads.append(filename)

print(f"\nDOWNLOAD SUMMARY:")
print(f"Successful: {len(successful_downloads)}/{len(dataset_files)}")
for file in successful_downloads:
    print(f"   - {os.path.join(dataset_dir, file)}")

if failed_downloads:
    for file in failed_downloads:
        print(f"   - {file}")

Files will be downloaded to: c:\Users\talha\Downloads\predict\yolov5\yolov5\dataset

DOWNLOAD SUMMARY:
Successful: 6/6
   - dataset\labels.zip
   - dataset\images.zip
   - dataset\train.txt
   - dataset\val.txt
   - dataset\dataset.yaml
   - dataset\best.pt


In [12]:
import zipfile
import os
import shutil

def extract_zip_file(zip_path, extract_to):
    """Extract zip file with error handling"""
    try:
        if not os.path.exists(zip_path):
            return False
        file_size = os.path.getsize(zip_path)
        if file_size < 100:
            return False
        os.makedirs(extract_to, exist_ok=True)

        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)

        extracted_count = 0
        for root, dirs, files in os.walk(extract_to):
            extracted_count += len(files)

        os.remove(zip_path)
        return True
    
    except:
        return False

dataset_dir = "./dataset"
extractions = [
    (os.path.join(dataset_dir, 'labels.zip'), os.path.join(dataset_dir, 'labels')),
    (os.path.join(dataset_dir, 'images.zip'), os.path.join(dataset_dir, 'images'))
]

extraction_success = True
for zip_file, target_dir in extractions:
    if not extract_zip_file(zip_file, target_dir):
        extraction_success = False

if extraction_success:
    labels_dir = os.path.join(dataset_dir, 'labels')
    images_dir = os.path.join(dataset_dir, 'images')
    
    if os.path.exists(labels_dir):
        label_files = [f for f in os.listdir(labels_dir) if f.endswith('.txt')]
        print(f"Label files: {len(label_files)}")
    
    if os.path.exists(images_dir):
        image_files = [f for f in os.listdir(images_dir) 
                      if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
        print(f"Image files: {len(image_files)}")

else:
    print(f"\nExtraction failed!")

Label files: 30000
Image files: 30000


In [13]:
import subprocess
import sys
import os

def verify_yolov5_setup():
    """Verify all required files are present"""
    print("VERIFYING YOLOV5 SETUP")
    print("*"*50)
    
    checks = {
        'YOLOv5 train.py': 'train.py',
        'Dataset config': './dataset/dataset.yaml', 
        'Training data': './dataset/train.txt',
        'Validation data': './dataset/val.txt',
        'Images directory': './dataset/images',
        'Labels directory': './dataset/labels'
    }
    
    all_good = True
    for name, path in checks.items():
        if os.path.exists(path):
            if os.path.isdir(path):
                count = len(os.listdir(path))
            else:
                size = os.path.getsize(path)
        else:
            all_good = False
    
    return all_good

def start_yolov5_training():
    """Start YOLOv5 training with optimal parameters"""
    print(f"\nSTARTING YOLOV5 TRAINING")
    print("*"*50)

    try:
        import torch
        if torch.cuda.is_available():
            gpu_name = torch.cuda.get_device_name(0)
            gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
            print(f"GPU: {gpu_name} ({gpu_memory:.1f} GB)")
            batch_size = 64
        else:
            batch_size = 32 
    except:
        batch_size = 32

    train_params = [
        sys.executable, 'train.py',
        '--img', '672',
        '--batch', str(batch_size),
        '--epochs', '100',
        '--data', 'dataset.yaml',
        '--weights', 'yolov5l.pt',
        '--project', 'runs/train',
        '--name', 'pid_experiment',
        '--cache',
        '--multi-scale'
    ]
    
    try:
        process = subprocess.Popen(train_params, 
                                stdout=subprocess.PIPE, 
                                stderr=subprocess.STDOUT,
                                universal_newlines=True,
                                bufsize=1)

        for line in process.stdout:
            print(line.strip())

        return_code = process.wait()
        
        if return_code == 0:
            return True
        else:
            return False
            
    except:
        return False

if verify_yolov5_setup():
    print(f"\nAll checks passed!")
    
else:
    print("check the errors above and fix them before training")

VERIFYING YOLOV5 SETUP
**************************************************

All checks passed!


In [14]:
import subprocess
import sys
import os
from pathlib import Path

def find_best_model():
    """Find the best trained model"""
    possible_paths = [
        'runs/train/pid_experiment/weights/best.pt',
        'runs/train/pid_experiment/weights/last.pt',
        './dataset/best.pt',
    ]
    
    for path in possible_paths:
        if os.path.exists(path):
            size = os.path.getsize(path) / 1024 / 1024  # MB
            print(f"Found model: {path} ({size:.1f} MB)")
            return path
    
    return None

def run_inference_test(model_path):
    """Test the model with inference"""

    input_image = r"C:\Users\talha\Downloads\predict\data\ss1.png"

    inference_cmd = [
        sys.executable, 'detect.py',
        '--weights', model_path,
        '--source', input_image,
        '--img', '640',
        '--conf', '0.25',
        '--iou', '0.45',
        '--save-txt',
        '--save-conf',
        '--project', 'runs/detect',
        '--name', 'pid_test'
    ]

    try:
        result = subprocess.run(inference_cmd, 
                            capture_output=True, 
                            text=True, 
                            timeout=300)
        if result.returncode == 0:

            output_lines = result.stdout.split('\n')
            for line in output_lines[-10:]:
                if line.strip():
                    print(f"   {line}")
            
            return True
        else:
            return False
    except Exception as e:
        return False
    
model_path = find_best_model()

if model_path:

    test_success = run_inference_test(model_path)
    

Found model: ./dataset/best.pt (354.4 MB)
